In [1]:
import pandas as pd
import re
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv('mental_health.csv')

print("First 5 rows of data:")
print(df.head())

print("\nHow many posts are label 0 (normal) vs label 1 (distress):")
print(df['label'].value_counts())

First 5 rows of data:
                                                text  label
0  dear american teens question dutch person hear...      0
1  nothing look forward lifei dont many reasons k...      1
2  music recommendations im looking expand playli...      0
3  im done trying feel betterthe reason im still ...      1
4  worried  year old girl subject domestic physic...      1

How many posts are label 0 (normal) vs label 1 (distress):
label
0    14139
1    13838
Name: count, dtype: int64


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [4]:
df['clean_text'] = df['text'].apply(clean_text)

print("\nExample of cleaned text:")
print("Before:", df['text'].iloc[34])
print("After :", df['clean_text'].iloc[0])


Example of cleaned text:
Before: cant go oni wanna kill ive gone shit cant bare life anymore im always shouted main person complain fucking shit self harm cuts armwrist cant tell anyone coz one mum dad always going want stop think redflag everyday probably leave everything soon need support reason live please help
After : dear american teens question dutch person heard guys get way easier things learn age us sooooo thth graders like right guys learn math


In [5]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

print("\nShape of our number table (rows, columns):", X.shape)
print("Example vocabulary words learned:", vectorizer.get_feature_names_out()[:10])



Shape of our number table (rows, columns): (27977, 5000)
Example vocabulary words learned: ['abandon' 'abandoned' 'abandoning' 'abandonment' 'abilities' 'ability'
 'able' 'abortion' 'about' 'above']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
model = LogisticRegression()
model.fit(X_train, y_train)

print("\nModel training complete!")


Model training complete!


In [8]:
predictions = model.predict(X_test)
print("\nPerformance report:")
print(classification_report(y_test, predictions))



Performance report:
              precision    recall  f1-score   support

           0       0.92      0.93      0.92      2802
           1       0.93      0.91      0.92      2794

    accuracy                           0.92      5596
   macro avg       0.92      0.92      0.92      5596
weighted avg       0.92      0.92      0.92      5596



In [9]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

word_importance = sorted(zip(feature_names, coefficients), key=lambda x: x[1], reverse=True)

print("\nTop 10 words pushing prediction toward DISTRESS (label 1):")
for word, score in word_importance[:10]:
    print(f"  {word}: {score:.3f}")

print("\nTop 10 words pushing prediction toward NORMAL (label 0):")
for word, score in word_importance[-10:]:
    print(f"  {word}: {score:.3f}")


Top 10 words pushing prediction toward DISTRESS (label 1):
  redflag: 8.050
  kill: 7.594
  die: 5.577
  suicidal: 5.568
  life: 5.135
  cannot: 4.923
  killing: 4.295
  depression: 4.226
  myself: 4.075
  alive: 3.933

Top 10 words pushing prediction toward NORMAL (label 0):
  school: -2.605
  horny: -2.632
  kinda: -2.682
  guys: -2.739
  yall: -2.847
  bored: -2.919
  crush: -3.779
  movie: -4.796
  film: -4.898
  br: -5.272


In [10]:
print("\nModel's bias/intercept (its fallback guess when input is all zeros):")
print(model.intercept_)


with open('tfidf_model.pkl', 'wb') as f:
    pickle.dump((vectorizer, model), f)

print("\nModel saved as tfidf_model.pkl — ready to use in the app!")


Model's bias/intercept (its fallback guess when input is all zeros):
[-2.04423212]

Model saved as tfidf_model.pkl — ready to use in the app!
